# NCA + kNN Classification Pipeline

In [1]:
import sys
from pathlib import Path

ABSOLUTE_PATH = Path().resolve()
PROJECT_ROOT  = ABSOLUTE_PATH.parents[2]
DATA_DIR      = PROJECT_ROOT / "data" / "raw"
ARCH_EVAL_DIR = ABSOLUTE_PATH / "arch-eval-results"
WEIGHTS_DIR   = ARCH_EVAL_DIR / "weights"

# cbam_end fold-0 weights — saved by arch-eval notebook
CBAM_END_WEIGHTS = WEIGHTS_DIR / "cbam_end" / "fold_0.pt"

sys.path.append(str(PROJECT_ROOT))

print(f"Project root  : {PROJECT_ROOT}")
print(f"Data dir      : {DATA_DIR}")
print(f"Weights       : {CBAM_END_WEIGHTS}")
print(f"Weights exist : {CBAM_END_WEIGHTS.exists()}")

Project root  : C:\Users\markm\Workspace\ms-machine-learning-diagnosis
Data dir      : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
Weights       : C:\Users\markm\Workspace\ms-machine-learning-diagnosis\src\notebooks\CNNS\arch-eval-results\weights\cbam_end\fold_0.pt
Weights exist : True


In [2]:
import numpy as np
from sklearn.preprocessing import StandardScaler

import src.scripts.data      as data
import src.scripts.models    as models
import src.scripts.trainer   as trainer
import src.scripts.evaluator as evaluator
import src.scripts.utils     as utils

utils.set_seed(42)

Random seed set to 42 for Python, NumPy, and PyTorch


## 1 · Data

In [5]:
path, categories = data.get_dataset(str(DATA_DIR))
classes = data.get_classes(path, categories, visualise=False)

image_paths, labels = data.get_paths_and_labels(path, classes)
train_transform, test_transform = data.get_transforms()

# Identical outer split to all other experiments
X_trainval, y_trainval, X_test, y_test = data.get_trainval_test_split(
    image_paths, labels,
    test_split=0.20,
    SEED=42
)

# Loaders
train_loader = data.get_test_loader(X_trainval, y_trainval, test_transform, batch_size=32)
test_loader  = data.get_test_loader(X_test,     y_test,     test_transform, batch_size=32)

print(f"Train+val : {len(X_trainval)} samples")
print(f"Test      : {len(X_test)} samples")

get_dataset()>>> Dataset already exists in C:\Users\markm\Workspace\ms-machine-learning-diagnosis\data\raw
get_dataset()>>> Available categories: ['Control Axial_crop', 'Control Saggital_crop', 'MS Axial_crop', 'MS Saggital_crop']
get_paths_and_labels()>>> Total images: 1652
get_trainval_test_split()>>> Train+Val pool : 1321 (80.0%)
get_trainval_test_split()>>> Held-out test  : 331 (20.0%)
get_trainval_test_split()>>> TrainVal class ratio — MS: 520  Non-MS: 801
get_trainval_test_split()>>> Test     class ratio — MS: 130  Non-MS: 201
get_test_loader()>>> Test loader ready — 1321 samples
get_test_loader()>>> Test loader ready — 331 samples
Train+val : 1321 samples
Test      : 331 samples


## 2 · Load Backbone

Loads the cnn backbone fold weights saved by the architecture evaluation. The linear classifier head is present in the loaded model but is never called — `return_features=True` routes the forward pass directly to the 512-dim avgpool output.

This run uses the `cbam_end` variation.

In [6]:
# Load backbone
import pandas as pd

OPTIMAL_HEAD_FILE = ABSOLUTE_PATH / "head-ablation-results" / "optimal_head.csv"
WINNING_HEAD = pd.read_csv(OPTIMAL_HEAD_FILE).iloc[0]["head"] if OPTIMAL_HEAD_FILE.exists() else "linear"
print(f"Head type : {WINNING_HEAD}")

model = models.get_model(architecture="cbam_end", head=WINNING_HEAD)
model = utils.load_weights(model, CBAM_END_WEIGHTS)
model.eval()
print("cbam_end loaded and set to eval mode.")

Head type : linear
get_model()>>> architecture='cbam_end'  head='linear'
load_weights()>>> Model loaded successfully and set to evaluation mode.
cbam_end loaded and set to eval mode.


## 3 · Feature Extraction

Passes all train+val and test images through the frozen backbone. The head is bypassed — `return_features=True` returns the raw 512-dim avgpool embedding for each image.

In [7]:
X_train_features, y_train_labels = trainer.get_features(model, train_loader)
X_test_features,  y_test_labels  = trainer.get_features(model, test_loader)

print(f"Train features : {X_train_features.shape}")
print(f"Test features  : {X_test_features.shape}")

Train features : (1321, 512)
Test features  : (331, 512)


## 4 · StandardScaler + NCA

**StandardScaler** — zero-mean, unit-variance normalisation. Fit on training features only, then applied to test features (no leakage).

**NCA** — learns a linear projection that maximises neighbourhood class separation. Reduces 512 → `TARGET_DIM` dimensions. Fit on training features only.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled  = scaler.transform(X_test_features)

print(f"Scaled train : mean≈{X_train_scaled.mean():.3f}  std≈{X_train_scaled.std():.3f}")
print(f"Scaled test  : mean≈{X_test_scaled.mean():.3f}  std≈{X_test_scaled.std():.3f}")

In [ ]:
TARGET_DIM = 256   # 512 → 256; adjust based on nca-knn-eval.ipynb parameter search

X_train_selected, X_test_selected = trainer.get_nca_features(
    X_train_scaled, y_train_labels,
    X_test_scaled,
    TARGET_DIM=TARGET_DIM
)

## 5 · kNN Classification

Distance-weighted kNN fitted on NCA-projected training features. `weights='distance'` is justified here because NCA has explicitly optimised the projection to make neighbourhood distances meaningful.

In [ ]:
NUM_NEIGHBOURS = 3   # adjust based on nca-knn-eval.ipynb parameter search

knn   = trainer.get_and_train_knn(X_train_selected, y_train_labels, NUM_NEIGHBOURS=NUM_NEIGHBOURS)
y_pred = evaluator.predict_knn(knn, X_test_selected)

print(f"Predictions shape : {y_pred.shape}")
print(f"Predicted positives (MS)     : {y_pred.sum()}")
print(f"Actual positives   (MS)      : {int(sum(y_test_labels))}")

## 6 · Evaluation

**Note on y_probs:** kNN does not produce calibrated sigmoid probabilities, so AUC-ROC and ECE are not reported here. The full SRQ2 evaluation in `nca-knn-eval.ipynb` handles this consistently across all backbones.

In [ ]:
evaluator.evaluate_model(
    y_true=y_test_labels,
    y_pred=y_pred,
    y_probs=None   # kNN has no sigmoid probabilities; AUC/ECE skipped
)